# Loading your BWK sample points.
## Loading packages
Let's start with installing the required packages.

In [111]:
try:
    import ee, geemap
except ImportError:
    !pip -q install earthengine-api geemap

import ee, geemap, datetime # ee: Google Earth Engine (GEE) API, geemap: mapping GEE products
import pandas as pd
import matplotlib.pyplot as plt # for figure plotting
import geopandas as gpd # for geospatial dataframes
import os

In [112]:
%pip install "leafmap[maplibre]" # For interactive visualization of maps.
from maplibre.plugins import MapboxDrawControls, MapboxDrawOptions # To select ROI's on your maps.

Connect to your Google Drive to load your dataset within your drive.

In [113]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
import ee
try:
    ee.Initialize(project="ee-sebastiaanverbesselt")
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

print('Earth Engine version:', ee.__version__)
print('geemap version:', geemap.__version__)


In [ ]:
import os
import ee
import geemap
import geopandas as gpd
import numpy as np
import pandas as pd

# 1. Base directories and Earth Engine setup
base_input_dir = "/content/drive/Shareddrives/PRJ_SlimmeBeeldverwerking/PRJ_2026_SlimmeBeeldverwerking/BWK - graslandfases/selecties"
output_dir = "/content/drive/Shareddrives/PRJ_SlimmeBeeldverwerking/PRJ_2026_SlimmeBeeldverwerking/Slimmebeeldverwerking/foundation_model_embeddings_grassland_phase_detection/OutputAlphaEarthEmbeddings"
os.makedirs(output_dir, exist_ok=True)

collection_id = "GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL"
inventory_years = [2019, 2020, 2021, 2022, 2023]


# Helper function to convert GeoDataFrame to Earth Engine FeatureCollection
def gdf_to_ee(gdf):
  features = []
  for _, row in gdf.iterrows():
    geom = ee.Geometry.Point([row.geometry.x, row.geometry.y])
    features.append(
        ee.Feature(
            geom,
            {
                "id": int(row["orig_id"]),
                "class": str(row["Level4"]),
                "lon": float(row.geometry.x),
                "lat": float(row.geometry.y),
            },
        )
    )
  return ee.FeatureCollection(features)


# 2. Main Processing Loop
for inv_year in inventory_years:
  print(f"\n{'='*55}\nProcessing Inventory Year: {inv_year}\n{'='*55}")

  grass_path = os.path.join(
      base_input_dir, f"Grassland_classified_final_classes_{inv_year}.gpkg"
  )
  nograss_path = os.path.join(
      base_input_dir, f"NoGrassland_classified_final_classes_{inv_year}.gpkg"
  )

  if not os.path.exists(grass_path) or not os.path.exists(nograss_path):
    print(f"Skipping {inv_year}: Dataset files not found.")
    continue

  # Load datasets
  grasslands = gpd.read_file(grass_path)
  nograsslands = gpd.read_file(nograss_path)

  # Sample 30% of nograsslands
  samplesize = round(len(grasslands) * 30 / 100)
  nograsslands_samp = nograsslands.sample(
      n=min(samplesize, len(nograsslands)), random_state=42
  )
  nograsslands_samp = nograsslands_samp.assign(Level4="nograss")
  nograsslands_samp = nograsslands_samp.to_crs(grasslands.crs)

  # Combine and convert multipolygons to centroids
  datagrassbwk = pd.concat([grasslands, nograsslands_samp], ignore_index=True)
  datagrassbwk["orig_id"] = datagrassbwk.index
  datagrassbwk["geometry"] = datagrassbwk.geometry.centroid

  # Project to WGS 84 (EPSG:4326)
  points_ee = datagrassbwk.to_crs("EPSG:4326")[
      ["orig_id", "Level4", "geometry"]
  ].copy()
  print(f"Total points to sample for {inv_year}: {len(points_ee)}")

  # Convert to EE FeatureCollection
  fc = gdf_to_ee(points_ee)

  # 3. Build a 3-year multi-band image (inv_year - 2, inv_year - 1, inv_year)
  emb_years = [inv_year - 2, inv_year - 1, inv_year]
  yearly_images = []

  for emb_year in emb_years:
    start_date = f"{emb_year}-01-01"
    end_date = f"{emb_year + 1}-01-01"

    # Filter annual mosaic
    img = (
        ee.ImageCollection(collection_id)
        .filterDate(start_date, end_date)
        .mosaic()
    )

    # Prefix bands by year (e.g. A00 -> A00_2017) to prevent collisions when combining
    band_names = img.bandNames()
    prefixed_band_names = band_names.map(
        lambda b: ee.String(b).cat(f"_{emb_year}")
    )
    img_prefixed = img.select(band_names, prefixed_band_names)
    yearly_images.append(img_prefixed)

  # Combine all 3 years into a single 192-band image
  combined_image = ee.Image.cat(yearly_images)

  # 4. Sample points across all 3 years in a single request
  print(f"Sampling 3-year composite embeddings for {emb_years}...")
  samples_fc = combined_image.sampleRegions(
      collection=fc, scale=10, geometries=True
  )

  # Stream directly into a GeoPandas GeoDataFrame
  embeddings_gdf = ee.data.computeFeatures(
      {"expression": samples_fc, "fileFormat": "GEOPANDAS_GEODATAFRAME"}
  )
  embeddings_gdf.crs = "EPSG:4326"

  # Add inventory year indicator
  embeddings_gdf["inventory_year"] = inv_year

  # Reorder columns: id, class, coordinates, inventory_year, followed by the embedding bands
  meta_cols = ["id", "class", "lon", "lat", "inventory_year", "geometry"]
  emb_cols = [c for c in embeddings_gdf.columns if c not in meta_cols]
  ordered_cols = [c for c in meta_cols if c in embeddings_gdf.columns] + sorted(
      emb_cols
  )
  embeddings_gdf = embeddings_gdf[ordered_cols]

  # Save to CSV
  out_csv = os.path.join(output_dir, f"AlphaEarth_inventory_{inv_year}_3yr.csv")
  embeddings_gdf.drop(columns=["geometry"]).to_csv(out_csv, index=False)
  print(f"Saved: {out_csv} | Shape: {embeddings_gdf.shape}")